# Fine-Tuning Fundamentals With the Hugging Face Trainer

**What you'll learn**
- Give a pretrained model a new head for your own labels
- Count, freeze and train parameters
- What one trainable parameter costs in GPU memory
- Train with the Hugging Face `Trainer`, score with `evaluate`
- Learning rate, epochs, overfitting, and stopping at the best epoch

## Setup
- Ten card intents from Banking77: the card problem Laya was fine-tuned on earlier
- `train_test_split` holds out 200 training messages for validation; `test` stays untouched

In [ ]:
import torch
import evaluate
import pandas as pd
from datasets import load_dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer,
                          EarlyStoppingCallback)

NAMES = [
    "card_arrival",
    "card_delivery_estimate",
    "activate_my_card",
    "card_not_working",
    "lost_or_stolen_card",
    "compromised_card",
    "card_payment_not_recognised",
    "declined_card_payment",
    "card_swallowed",
    "contactless_not_working",
]

In [ ]:
banking = load_dataset("mteb/banking77")  # 77 intents, real customer messages

def is_card_intent(row):
    return row["label_text"] in NAMES  # one of our ten card intents?

cards = banking.filter(is_card_intent)
cards

In [ ]:
def add_label(row):
    return {"label": NAMES.index(row["label_text"])}  # the intent's position in NAMES, 0 to 9

cards = cards.map(add_label)
cards["train"][0]

In [ ]:
split = cards["train"].train_test_split(test_size=200, seed=0)  # 200 training messages kept for validation
split

In [ ]:
train = split["train"]
val = split["test"]
test = cards["test"]
len(train), len(val), len(test)

## Add a Classification Head to a Pretrained Model
- `AutoModelForSequenceClassification`: the pretrained body plus one new linear layer, one output per label
- The warning names `score.weight`: the only layer that starts random
- `pad_token_id`: the head reads the last real token, so it must know which tokens are padding
- `float32` weights: the updates `Trainer` makes stay exact, the forward pass still runs in 16-bit

In [ ]:
MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
tok = AutoTokenizer.from_pretrained(MODEL)

def new_model():
    return AutoModelForSequenceClassification.from_pretrained(
        MODEL,
        num_labels=10,  # one output per card intent
        pad_token_id=tok.pad_token_id,  # so the head can find the last real token
        dtype=torch.float32,  # exact weight updates
    )

In [ ]:
model = new_model()
model.score

## Count Trainable and Frozen Parameters
- `requires_grad=True` on a tensor: the optimizer updates it

In [ ]:
model.num_parameters(), model.num_parameters(only_trainable=True)

In [ ]:
one_layer = 0
for p in model.model.layers[0].parameters():
    one_layer += p.numel()  # every weight in the first layer
one_layer

In [ ]:
parts = {
    "embeddings": model.model.embed_tokens.weight.numel(),
    "one layer": one_layer,
    "layers": len(model.model.layers),
    "new head": model.score.weight.numel(),
}
parts

## GPU Memory per Trainable Parameter
- Training keeps four numbers per trainable parameter: the weight, its gradient, AdamW's two running averages
- In `float32`: 4 + 4 + 4 + 4 = 16 bytes, before activations

In [ ]:
GB = 1024**3
n = model.num_parameters()
n

In [ ]:
memory = {
    "weights only": n * 4 / GB,  # 4 bytes per weight
    "full fine-tune": n * 16 / GB,  # weight, gradient and two AdamW averages
    "a 4B model, full fine-tune": 4e9 * 16 / GB,
}
memory

## Set Up the Hugging Face Trainer
- Tokenize once with `map`; `Trainer` pads each batch
- `compute_metrics`: accuracy from `evaluate`
- bf16 where the GPU supports it, fp16 on a free T4

In [ ]:
def tokenize(batch):
    return tok(batch["text"])  # token ids for a batch of messages

tokenize(train[:2])

In [ ]:
train = train.map(tokenize, batched=True)
val = val.map(tokenize, batched=True)
test = test.map(tokenize, batched=True)
train.column_names

In [ ]:
accuracy = evaluate.load("accuracy")
accuracy.compute(predictions=[0, 1, 2], references=[0, 1, 1])  # two of three right

In [ ]:
def compute_metrics(p):
    predictions = p.predictions.argmax(-1)  # the top label for each message
    return accuracy.compute(predictions=predictions, references=p.label_ids)

In [ ]:
BF16 = torch.cuda.is_bf16_supported()  # can this GPU train in bfloat16?
BF16

In [ ]:
def trainer(model, lr, epochs=3):
    args = TrainingArguments(
        "runs/fundamentals",
        learning_rate=lr,
        num_train_epochs=epochs,
        per_device_train_batch_size=16,
        bf16=BF16,  # 16-bit forward pass where the GPU supports bf16
        fp16=not BF16,  # otherwise fp16, as on a free T4
        eval_strategy="epoch",  # score the validation set after every epoch
        logging_strategy="epoch",
        save_strategy="no",
        report_to="none",
    )
    return Trainer(
        model=model,
        args=args,
        train_dataset=train,
        eval_dataset=val,
        processing_class=tok,  # pads each batch
        compute_metrics=compute_metrics,
    )

## Baseline Accuracy Before Training
- The new head is random: about one answer in ten is right

In [ ]:
baseline = trainer(model, 2e-5)
baseline.evaluate(test)

## Option 1: Freeze the Model and Train Only the Head
- Freeze the body, only `score` learns
- A new layer takes a much larger learning rate

In [ ]:
for p in model.model.parameters():
    p.requires_grad = False  # the body stays as it is
model.num_parameters(only_trainable=True)

In [ ]:
torch.cuda.reset_peak_memory_stats()
head = trainer(model, 1e-3)
head.train()

In [ ]:
scores = head.evaluate(test)
scores

In [ ]:
torch.cuda.max_memory_allocated() / GB  # the most GPU memory training used

## Option 2: Fine-Tune Every Weight
- A fresh copy, every weight trainable
- A small learning rate: the body already knows language

In [ ]:
del baseline, head, model
model = new_model()
torch.cuda.reset_peak_memory_stats()
full = trainer(model, 2e-5)
full.train()

In [ ]:
scores = full.evaluate(test)
scores

In [ ]:
torch.cuda.max_memory_allocated() / GB

## Accuracy vs Learning Rate
- The same full fine-tune at three learning rates

In [ ]:
del full, model
results = []
for lr in [1e-6, 2e-5, 1e-3]:
    model = new_model()  # a fresh model each time
    t = trainer(model, lr)
    t.train()
    scores = t.evaluate(test)
    results.append({"learning rate": lr, "accuracy": scores["eval_accuracy"]})

results

In [ ]:
pd.DataFrame(results)

## Overfitting: Training Loss vs Validation Loss
- Ten epochs, validation loss after each one
- Training loss keeps falling; validation loss turns back up

In [ ]:
del t, model
model = new_model()
long = trainer(model, 2e-5, epochs=10)
long.train()

In [ ]:
log = pd.DataFrame(long.state.log_history)  # one row per log entry, training and validation apart
log.head()

In [ ]:
log = log.groupby("epoch").first()  # one row per epoch
log[["loss", "eval_loss", "eval_accuracy"]]

In [ ]:
log[["loss", "eval_loss"]].plot(xlabel="epoch", ylabel="loss");

## Early Stopping: Stop When Validation Loss Rises
- Save after every epoch, keep the one with the lowest validation loss
- `EarlyStoppingCallback` ends the run after two epochs without improvement

In [ ]:
del long, model
model = new_model()

args = TrainingArguments(
    "runs/fundamentals",
    learning_rate=2e-5,
    num_train_epochs=10,
    per_device_train_batch_size=16,
    bf16=BF16,
    fp16=not BF16,
    eval_strategy="epoch",
    logging_strategy="epoch",
    save_strategy="epoch",  # a checkpoint after every epoch
    load_best_model_at_end=True,  # reload the best checkpoint when training ends
    metric_for_best_model="eval_loss",  # best means the lowest validation loss
    save_total_limit=1,  # keep only the best checkpoint on disk
    report_to="none",
)

In [ ]:
best = Trainer(
    model=model,
    args=args,
    train_dataset=train,
    eval_dataset=val,
    processing_class=tok,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],  # stop after two epochs without improvement
)

In [ ]:
best.train()

In [ ]:
best.state.best_model_checkpoint

In [ ]:
best.evaluate(test)["eval_accuracy"]

## Cheat Sheet
| Need | Code |
|---|---|
| New head for your labels | `AutoModelForSequenceClassification.from_pretrained(name, num_labels=n)` |
| Count | `model.num_parameters(only_trainable=True)` |
| Freeze | `p.requires_grad = False` |
| Memory, full fine-tune | about 16 bytes per parameter, plus activations |
| Train | `Trainer(model, TrainingArguments(...), train_dataset, eval_dataset, processing_class=tok)` |
| Metric | `evaluate.load("accuracy")` in `compute_metrics` |
| Precision | `bf16=True`, or `fp16=True` on a T4 |
| Overfitting | `eval_strategy="epoch"`, watch `eval_loss` |
| Keep the best epoch | `load_best_model_at_end=True`, `EarlyStoppingCallback` |